In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_recall_fscore_support
from torchvision import transforms

CURRENT_DIR = Path.cwd().resolve()
project_options = [
    CURRENT_DIR,
    CURRENT_DIR / "plant-disease-classification",
    CURRENT_DIR.parent,
]
PROJECT_DIR = next(
    (path for path in project_options if (path / "src/data_utils.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục project.")

sys.path.insert(0, str(PROJECT_DIR / "src"))

from data_utils import DATA_DIR, make_dataloaders

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook Complex CNN yêu cầu GPU CUDA.")

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"
MODELS_DIR = PROJECT_DIR / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

split_df = pd.read_csv(RESULTS_DIR / "data_split.csv")

with (RESULTS_DIR / "class_names.json").open(encoding="utf-8") as file:
    class_names = json.load(file)

num_classes = len(class_names)

IMAGE_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

print("Classes:", num_classes)
print(split_df["split"].value_counts())

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_loader, val_loader, test_loader = make_dataloaders(
    split_df=split_df,
    data_dir=DATA_DIR,
    train_transform=image_transform,
    eval_transform=image_transform,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

def prepare_batch(images, labels, training=False):
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    if training:
        flip_mask = torch.rand(
            len(images), 1, 1, 1, device=device
        ) < 0.5
        images = torch.where(
            flip_mask,
            images.flip(3),
            images,
        )

        brightness = 0.85 + 0.30 * torch.rand(
            len(images), 1, 1, 1, device=device
        )
        images = (images * brightness).clamp(0, 1)

    return images, labels

In [ ]:
class ParallelBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.branch_1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        self.branch_2 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
        )

        self.branch_3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        self.branch_4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, 16, kernel_size=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
        )

    def forward(self, x):
        x1 = self.branch_1(x)
        x2 = self.branch_2(x)
        x3 = self.branch_3(x)
        x4 = self.branch_4(x)

        return torch.cat([x1, x2, x3, x4], dim=1)

In [ ]:
class ComplexCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Parallel block
            ParallelBlock(64),

            # Block 3
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
model = ComplexCNN(num_classes).to(device)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

with torch.no_grad():
    sample = torch.zeros(
        2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device
    )
    assert model(sample).shape == (2, num_classes)

print(model)
print(f"Parameters: {total_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

In [ ]:
def train_one_epoch(model, dataloader):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = prepare_batch(
            images,
            labels,
            training=True,
        )

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(labels)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += len(labels)

    return running_loss / total, correct / total

In [ ]:
def evaluate(model, dataloader, return_predictions=False):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    predictions_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = prepare_batch(images, labels)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(1)

            running_loss += loss.item() * len(labels)
            correct += (predictions == labels).sum().item()
            total += len(labels)

            if return_predictions:
                predictions_list.append(predictions.cpu())
                labels_list.append(labels.cpu())

    result = (running_loss / total, correct / total)

    if return_predictions:
        result += (
            torch.cat(predictions_list).numpy(),
            torch.cat(labels_list).numpy(),
        )

    return result

In [ ]:
best_val_loss = float("inf")
best_epoch = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
    )

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

model.load_state_dict(best_state)
print("Best epoch:", best_epoch)

In [ ]:
test_loss, test_accuracy, predictions, labels = evaluate(
    model,
    test_loader,
    return_predictions=True,
)

precision, recall, f1_score, _ = precision_recall_fscore_support(
    labels,
    predictions,
    average="weighted",
    zero_division=0,
)

print(f"Test Loss : {test_loss:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1_score:.4f}")

In [ ]:
torch.save(
    {
        "model_state_dict": best_state,
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
    },
    MODELS_DIR / "complex_cnn_best.pt",
)

print("Saved:", MODELS_DIR / "complex_cnn_best.pt")